# 02 — Exploratory analysis of Ghanaian banking asset quality

This notebook replaces the earlier exploratory workflow with a reproducible, source-first audit of monthly NPL and macro-financial indicators. It documents missingness, uses transparent transformations, and runs preliminary time-series diagnostics.

- The main sample is the cleaned monthly dataset exported by `01_data_cleaning.ipynb` from `data/raw/data_first.csv` (January 2010–December 2025).
- The main sample comes from `data/raw/data_first.csv` (monthly observations, January 2010–December 2025).
- Quarterly GDP is deliberately excluded from this monthly notebook. The existing monthly GDP series is linearly interpolated from quarterly observations and is not observed monthly GDP.
- The BoG July, September and November 2026 summary files are draft publication vintages and are not spliced into this historical sample. Their definitions, vintage consistency and source transcription should be verified before combining them.
- This is descriptive analysis, not causal inference. ADF tests are preliminary and do not account for structural breaks.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

# Resolve the repository root whether this notebook is launched from the root or notebooks/.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / 'data' / 'raw' / 'data_first.csv').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Could not locate data/raw/data_first.csv from the current working directory.')

RAW_FILE = ROOT / 'data' / 'raw' / 'data_first.csv'
OUT_DIR = ROOT / 'data' / 'processed'
FIG_DIR = ROOT / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)


## 1. Load and validate the monthly source

The source file contains metadata rows after the observations. Parse the date column and retain only rows with valid dates; do not rely on a fixed row count.

In [ ]:
monthly = pd.read_csv(OUT_DIR / 'ghana_banking_monthly_clean.csv', parse_dates=['Date'], index_col='Date')
monthly = monthly.sort_index().apply(pd.to_numeric, errors='coerce')

if monthly.index.has_duplicates:
    raise ValueError('Duplicate monthly dates found in the cleaned dataset.')
if not monthly.index.is_monotonic_increasing:
    raise ValueError('Monthly observations are not in chronological order.')

expected = [
    'Monetary Policy Rate (%)', 'Consumer Price Index, All Items',
    'USD Exchange Rate, monthly averages', 'Total Liquidity (M2+)',
    'Gold Price (Realised Gold Price)', 'Return on Assets',
    'Non Performing Loan Ratio', 'Capital Adequacy Ratio'
]
missing_columns = sorted(set(expected) - set(monthly.columns))
if missing_columns:
    raise ValueError(f'Missing expected variables: {missing_columns}')

print(f'Sample: {monthly.index.min():%Y-%m} to {monthly.index.max():%Y-%m}')
print(f'Rows: {len(monthly):,}; columns: {monthly.shape[1]}')
monthly.tail()


## 2. Data-quality audit

Missing observations are reported rather than automatically filled. In particular, do not forward-fill the end-of-sample missing gold and liquidity observations just to make a balanced table.

In [ ]:
audit = pd.DataFrame({
    'non_missing': monthly.notna().sum(),
    'missing': monthly.isna().sum(),
    'missing_pct': 100 * monthly.isna().mean(),
    'first_valid': monthly.apply(lambda s: s.first_valid_index()),
    'last_valid': monthly.apply(lambda s: s.last_valid_index()),
    'minimum': monthly.min(),
    'maximum': monthly.max(),
})
display(audit)
display(monthly[expected].describe().T)

if (monthly['Non Performing Loan Ratio'].dropna() < 0).any():
    raise ValueError('NPL ratio contains negative values; inspect the source.')
if (monthly['Consumer Price Index, All Items'].dropna() <= 0).any():
    raise ValueError('CPI must be positive before calculating log changes.')
if (monthly['USD Exchange Rate, monthly averages'].dropna() <= 0).any():
    raise ValueError('Exchange rate must be positive before calculating log changes.')
if (monthly['Total Liquidity (M2+)'].dropna() <= 0).any():
    raise ValueError('M2+ must be positive before calculating log changes.')


## 3. Plot the main series

The first figure shows the NPL ratio and the selected macro-financial indicators in their published units. The panels have separate vertical scales, so compare time patterns rather than visual magnitudes across panels.

In [ ]:
plot_columns = [
    'Non Performing Loan Ratio', 'Capital Adequacy Ratio', 'Return on Assets',
    'Monetary Policy Rate (%)', 'Consumer Price Index, All Items',
    'USD Exchange Rate, monthly averages', 'Total Liquidity (M2+)'
]
fig, axes = plt.subplots(len(plot_columns), 1, figsize=(13, 18), sharex=True)
for ax, col in zip(axes, plot_columns):
    ax.plot(monthly.index, monthly[col], linewidth=1.4)
    ax.set_ylabel(col, fontsize=8)
    ax.grid(alpha=0.25)
axes[-1].set_xlabel('Date')
fig.suptitle('Ghana: monthly banking and macro-financial indicators', y=1.002)
fig.tight_layout()
fig.savefig(FIG_DIR / 'monthly_indicator_timeseries.png', dpi=160, bbox_inches='tight')
plt.show()


## 4. Construct transparent monthly transformations

For a first-pass analysis, retain the NPL ratio, CAR, ROA and policy rate in levels. Convert CPI, the GHS-per-USD exchange rate and M2+ into monthly log changes multiplied by 100. A positive exchange-rate change means the cedi depreciated against the US dollar. These are monthly percentage changes, **not annualised rates**.

In [ ]:
transformed = pd.DataFrame(index=monthly.index)
transformed['NPL_ratio_pct'] = monthly['Non Performing Loan Ratio']
transformed['CAR_pct'] = monthly['Capital Adequacy Ratio']
transformed['ROA_pct'] = monthly['Return on Assets']
transformed['Policy_rate_pct'] = monthly['Monetary Policy Rate (%)']
transformed['Inflation_mom_pct'] = 100 * np.log(monthly['Consumer Price Index, All Items']).diff()
transformed['Cedi_depreciation_mom_pct'] = 100 * np.log(monthly['USD Exchange Rate, monthly averages']).diff()
transformed['M2_growth_mom_pct'] = 100 * np.log(monthly['Total Liquidity (M2+)']).diff()

display(transformed.describe().T)
display(transformed.isna().sum().rename('missing observations').to_frame())
transformed.to_csv(OUT_DIR / 'monthly_eda_dataset.csv', index_label='Date')


## 5. Explore changes and pairwise correlations

Correlations are descriptive and do not establish direction of influence. The levels correlation matrix can be dominated by common trends; the transformed matrix is often more informative, but it is not a substitute for a properly specified time-series model.

In [ ]:
corr_cols = list(transformed.columns)
corr_levels = transformed[corr_cols].corr()
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr_levels, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(len(corr_cols)), corr_cols, rotation=70, ha='right')
ax.set_yticks(range(len(corr_cols)), corr_cols)
fig.colorbar(im, ax=ax, label='Pearson correlation')
ax.set_title('Pairwise correlations: levels and monthly changes')
fig.tight_layout()
fig.savefig(FIG_DIR / 'monthly_eda_correlation.png', dpi=160, bbox_inches='tight')
plt.show()

display(corr_levels.round(2))


## 6. Preliminary unit-root checks

The Augmented Dickey–Fuller (ADF) test is run on selected levels and first differences using an intercept and AIC-based lag selection. The null hypothesis is a unit root. Failure to reject is not proof of non-stationarity; results can be sensitive to deterministic terms, lag choice, sample period and structural breaks. Treat this table as a diagnostic starting point, not the final stationarity decision.

In [ ]:
adf_series = {
    'NPL ratio (level)': transformed['NPL_ratio_pct'],
    'NPL ratio (first difference)': transformed['NPL_ratio_pct'].diff(),
    'CPI inflation (monthly log change)': transformed['Inflation_mom_pct'],
    'Cedi depreciation (monthly log change)': transformed['Cedi_depreciation_mom_pct'],
    'M2 growth (monthly log change)': transformed['M2_growth_mom_pct'],
    'Policy rate (level)': transformed['Policy_rate_pct'],
    'CAR (level)': transformed['CAR_pct'],
    'ROA (level)': transformed['ROA_pct'],
}
adf_rows = []
for name, series in adf_series.items():
    x = series.dropna()
    if len(x) < 24 or x.nunique() < 2:
        continue
    result = adfuller(x, regression='c', autolag='AIC')
    adf_rows.append({
        'series': name, 'nobs': result[3], 'ADF statistic': result[0],
        'p-value': result[1], 'used lags': result[2],
        '1% critical value': result[4]['1%'],
        '5% critical value': result[4]['5%'],
    })
adf_results = pd.DataFrame(adf_rows).set_index('series')
display(adf_results)
adf_results.to_csv(OUT_DIR / 'adf_preliminary_results.csv')


## 7. Export and interpretation checklist

The notebook writes a transformed monthly dataset and preliminary ADF results to `data/processed/`, and saves figures under `figures/`.

Before moving to regression:
1. Confirm the source definitions and whether the NPL ratio is comparable throughout the sample.
2. Investigate the unusual M2+ movements visible in the source data; do not assume every movement is an economic shock rather than a data issue or rebasing effect.
3. Check structural breaks, especially around major banking-sector reforms and the 2022–2023 macro-financial crisis.
4. Examine serial correlation and persistence in NPLs.
5. Decide the estimation sample after documenting missingness; do not mechanically impute missing end-of-sample values.
6. Only then specify a parsimonious dynamic model and compare it with sensible alternatives.

**No regression coefficients or causal claims are produced in this notebook.**